# Olist Brazilian E-Commerce — Data Cleaning

## Objective
Clean and standardize the nine raw Olist datasets while preserving the original source data.

## Cleaning Approach
1. Load the original datasets.
2. Handle missing values based on business context.
3. Remove exact duplicate records where appropriate.
4. Standardize data types and text values.
5. Investigate suspicious and inconsistent records.
6. Validate the cleaned datasets.
7. Export the cleaned datasets to the Processed folder.

**Important:** Original raw datasets will not be modified.

In [1]:
import pandas as pd
import numpy as np
import os

RAW_PATH = r"C:\Users\priya\OneDrive\Desktop\Documents\Downloads\Olist_Data_Warehouse\Dataset\Raw"
PROCESSED_PATH = r"C:\Users\priya\OneDrive\Desktop\Documents\Downloads\Olist_Data_Warehouse\Dataset\Processed"

os.makedirs(PROCESSED_PATH, exist_ok=True)

print("Libraries imported successfully!")
print("Output folder is ready!")

Libraries imported successfully!
Output folder is ready!


In [2]:
orders = pd.read_csv(RAW_PATH + r"\olist_orders_dataset.csv")
order_items = pd.read_csv(RAW_PATH + r"\olist_order_items_dataset.csv")
customers = pd.read_csv(RAW_PATH + r"\olist_customers_dataset.csv")
products = pd.read_csv(RAW_PATH + r"\olist_products_dataset.csv")
sellers = pd.read_csv(RAW_PATH + r"\olist_sellers_dataset.csv")
payments = pd.read_csv(RAW_PATH + r"\olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW_PATH + r"\olist_order_reviews_dataset.csv")
geolocation = pd.read_csv(RAW_PATH + r"\olist_geolocation_dataset.csv")
category_translation = pd.read_csv(RAW_PATH + r"\product_category_name_translation.csv")

print("All nine datasets loaded successfully!")

All nine datasets loaded successfully!


## Initial Dataset Inventory

Before applying cleaning operations, record the row and column counts of all nine datasets.

This provides a baseline to compare against the cleaned datasets.

In [7]:
dataset_inventory = pd.DataFrame({
    "Dataset": [
        "Orders", "Order Items", "Customers",
        "Products", "Sellers", "Payments",
        "Reviews", "Geolocation", "Category Translation"
    ],
    "Rows": [
        orders.shape[0],
        order_items.shape[0],
        customers.shape[0],
        products.shape[0],
        sellers.shape[0],
        payments.shape[0],
        reviews.shape[0],
        geolocation.shape[0],
        category_translation.shape[0]
    ],
    "Columns": [
        orders.shape[1],
        order_items.shape[1],
        customers.shape[1],
        products.shape[1],
        sellers.shape[1],
        payments.shape[1],
        reviews.shape[1],
        geolocation.shape[1],
        category_translation.shape[1]
    ]
})

dataset_inventory

,Dataset,Rows,Columns
0,Orders,99441,8
1,Order Items,112650,7
2,Customers,99441,5
3,Products,32951,9
4,Sellers,3095,4
5,Payments,103886,5
6,Reviews,99224,7
7,Geolocation,1000163,5
8,Category Translation,71,2


In [8]:
orders_clean = orders.copy()

In [9]:
orders_clean.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [10]:
orders_clean["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [11]:
print("Missing approval timestamp by status:")

print(
    orders_clean[
        orders_clean["order_approved_at"].isnull()
    ]["order_status"].value_counts()
)

print("\nMissing carrier timestamp by status:")

print(
    orders_clean[
        orders_clean["order_delivered_carrier_date"].isnull()
    ]["order_status"].value_counts()
)

print("\nMissing customer delivery timestamp by status:")

print(
    orders_clean[
        orders_clean["order_delivered_customer_date"].isnull()
    ]["order_status"].value_counts()
)

Missing approval timestamp by status:
order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

Missing carrier timestamp by status:
order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

Missing customer delivery timestamp by status:
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


In [12]:
# Convert order timestamp columns into datetime format

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders_clean[column] = pd.to_datetime(
        orders_clean[column],
        errors="coerce"
    )

print(orders_clean[date_columns].dtypes)

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [13]:
orders_clean.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [14]:
orders_clean.shape

(99441, 8)

In [15]:
invalid_delivery = orders_clean[
    orders_clean["order_delivered_customer_date"] <
    orders_clean["order_delivered_carrier_date"]
].copy()

print("Inconsistent delivery records:", len(invalid_delivery))

Inconsistent delivery records: 23


In [16]:
invalid_delivery[
    [
        "order_id",
        "order_status",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
].head(10)

,order_id,order_status,order_delivered_carrier_date,order_delivered_customer_date
6437,a1abeb653a4d4cd1e142ccb8c82cd069,delivered,2017-07-28 16:57:58,2017-07-25 19:32:56
9553,383aa8b2724fe452d9ccd9934a8c628b,delivered,2017-07-07 17:22:41,2017-07-06 14:27:51
13487,cb1134f9010d242e9515ad1c78ec0c39,delivered,2017-07-20 19:22:02,2017-07-19 14:13:28
14474,dceb62e8fa94b46006c9554fed743df0,delivered,2017-08-01 18:23:30,2017-07-26 18:09:10
19268,5f9d46795c3126674e52becb3a1a517f,delivered,2017-07-20 23:03:42,2017-07-20 18:52:41
21338,8c78d01de3a9009e23d6877a7cc9be20,delivered,2016-10-26 11:41:53,2016-10-25 17:51:46
22520,b27af682321527a6349f1761eb3f360c,delivered,2017-06-27 14:51:54,2017-06-26 15:45:35
25393,1cc3ae63caffff2d6c3ee3e78e074acf,delivered,2017-08-10 18:28:56,2017-08-10 18:05:38
25646,e37f11cae9985ca58f0b56f268720537,delivered,2017-08-01 18:17:47,2017-07-31 17:49:56
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,delivered,2017-08-09 18:18:43,2017-08-01 21:13:01


In [17]:
invalid_delivery["time_difference"] = (
    invalid_delivery["order_delivered_carrier_date"] -
    invalid_delivery["order_delivered_customer_date"]
)

invalid_delivery[
    ["order_id", "time_difference"]
].head(10)

,order_id,time_difference
6437,a1abeb653a4d4cd1e142ccb8c82cd069,2 days 21:25:02
9553,383aa8b2724fe452d9ccd9934a8c628b,1 days 02:54:50
13487,cb1134f9010d242e9515ad1c78ec0c39,1 days 05:08:34
14474,dceb62e8fa94b46006c9554fed743df0,6 days 00:14:20
19268,5f9d46795c3126674e52becb3a1a517f,0 days 04:11:01
21338,8c78d01de3a9009e23d6877a7cc9be20,0 days 17:50:07
22520,b27af682321527a6349f1761eb3f360c,0 days 23:06:19
25393,1cc3ae63caffff2d6c3ee3e78e074acf,0 days 00:23:18
25646,e37f11cae9985ca58f0b56f268720537,1 days 00:27:51
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,7 days 21:05:42


In [18]:
invalid_delivery["time_difference"].describe()

count                           23
mean     3 days 06:27:18.478260869
std      3 days 17:18:29.050924326
min                0 days 00:23:18
25%                0 days 23:20:18
50%                1 days 15:51:52
75%         5 days 08:45:32.500000
max               16 days 02:18:29
Name: time_difference, dtype: object

In [19]:
print("Original rows:", orders.shape[0])
print("Cleaned rows:", orders_clean.shape[0])

print(
    "Duplicate order IDs:",
    orders_clean["order_id"].duplicated().sum()
)

Original rows: 99441
Cleaned rows: 99441
Duplicate order IDs: 0


In [20]:
order_items_clean = order_items.copy()

In [21]:
order_items_clean.isnull().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [22]:
order_items_clean.duplicated().sum()

np.int64(0)

In [23]:
order_items_clean.duplicated(
    subset=["order_id", "order_item_id"]
).sum()

np.int64(0)

In [24]:
order_items_clean.dtypes

order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

In [25]:
order_items_clean[["price", "freight_value"]].describe()

,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


In [26]:
print("Zero price:", (order_items_clean["price"] == 0).sum())

print("Negative price:", (order_items_clean["price"] < 0).sum())

print("Zero freight:", (order_items_clean["freight_value"] == 0).sum())

print("Negative freight:", (order_items_clean["freight_value"] < 0).sum())

Zero price: 0
Negative price: 0
Zero freight: 383
Negative freight: 0


In [27]:
order_items_clean["shipping_limit_date"] = pd.to_datetime(
    order_items_clean["shipping_limit_date"],
    errors="coerce"
)

In [28]:
order_items_clean.dtypes

order_id                       object
order_item_id                   int64
product_id                     object
seller_id                      object
shipping_limit_date    datetime64[ns]
price                         float64
freight_value                 float64
dtype: object

In [29]:
zero_freight_items = order_items_clean[
    order_items_clean["freight_value"] == 0
]

print("Total zero-freight records:", len(zero_freight_items))

zero_freight_items[
    ["order_id", "order_item_id", "product_id",
     "seller_id", "price", "freight_value"]
].head(10)

Total zero-freight records: 383


,order_id,order_item_id,product_id,seller_id,price,freight_value
114,00404fa7a687c8c44ca69d42695aae73,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,99.9,0.0
258,00a870c6c06346e85335524935c600c0,1,aca2eb7d00ea1a7b8ebd4e68314663af,955fee9216a65b617aa5c0531780ce60,69.9,0.0
483,011c899816ea29773525bd3322dbb6aa,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,99.9,0.0
508,012b3f6ab7776a8ab3443a4ad7bef2e6,1,422879e10f46682990de24d770e7f83d,1f50f920176fa81dab994f9023523100,53.9,0.0
509,012b3f6ab7776a8ab3443a4ad7bef2e6,2,422879e10f46682990de24d770e7f83d,1f50f920176fa81dab994f9023523100,53.9,0.0
1784,04105b54650921ca3246f52e6f175f46,1,aca2eb7d00ea1a7b8ebd4e68314663af,955fee9216a65b617aa5c0531780ce60,69.9,0.0
2232,0517a3e68dac3308995edca2144db36e,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,99.9,0.0
2714,061ba2e2d7544790b6ed6b5b4dd9278c,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,110.0,0.0
3224,07441f525824bc6b31d4dc19c5d49fc9,1,53b36df67ebb7c41585e8d54d6772e08,4869f7a5dfa277a7dca6462dcf3b52b2,106.9,0.0
3378,079f16689c29acb6cab92978e6af2137,1,aca2eb7d00ea1a7b8ebd4e68314663af,955fee9216a65b617aa5c0531780ce60,69.9,0.0


In [30]:
print("Original rows:", len(order_items))
print("Cleaned rows:", len(order_items_clean))

print("Missing values:", order_items_clean.isnull().sum().sum())

print("Duplicate order-item keys:",
      order_items_clean.duplicated(
          subset=["order_id", "order_item_id"]
      ).sum())

print("Zero freight records:",
      (order_items_clean["freight_value"] == 0).sum())

Original rows: 112650
Cleaned rows: 112650
Missing values: 0
Duplicate order-item keys: 0
Zero freight records: 383


In [31]:
order_items_clean.to_csv(
    os.path.join(PROCESSED_PATH, "order_items_clean.csv"),
    index=False
)

In [32]:
customers_clean = customers.copy()

In [33]:
customers_clean.isnull().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [34]:
print("Exact duplicate rows:",
      customers_clean.duplicated().sum())

print("Duplicate customer IDs:",
      customers_clean.duplicated(
          subset=["customer_id"]
      ).sum())

Exact duplicate rows: 0
Duplicate customer IDs: 0


In [35]:
customers_clean.dtypes

customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

In [36]:
customers_clean["customer_zip_code_prefix"] = (
    customers_clean["customer_zip_code_prefix"].astype(str)
)

In [37]:
customers_clean.dtypes

customer_id                 object
customer_unique_id          object
customer_zip_code_prefix    object
customer_city               object
customer_state              object
dtype: object

In [38]:
print("Missing ZIP codes:",
      customers_clean["customer_zip_code_prefix"].isnull().sum())

print("ZIP codes with unexpected length:",
      (~customers_clean["customer_zip_code_prefix"].str.len().eq(5)).sum())

print("Unique ZIP codes:",
      customers_clean["customer_zip_code_prefix"].nunique())

print("Sample ZIP codes:")
print(customers_clean["customer_zip_code_prefix"].head(10))

Missing ZIP codes: 0
ZIP codes with unexpected length: 23995
Unique ZIP codes: 14994
Sample ZIP codes:
0    14409
1     9790
2     1151
3     8775
4    13056
5    89254
6     4534
7    35182
8    81560
9    30575
Name: customer_zip_code_prefix, dtype: object


In [39]:
customers_clean["customer_zip_code_prefix"].str.len().value_counts().sort_index()

customer_zip_code_prefix
4    23995
5    75446
Name: count, dtype: int64

In [40]:
customers_clean["customer_zip_code_prefix"] = (
    customers_clean["customer_zip_code_prefix"].str.zfill(5)
)

In [41]:
print("ZIP code length distribution:")
print(customers_clean["customer_zip_code_prefix"].str.len().value_counts())

print("Unexpected ZIP code lengths:",
      (~customers_clean["customer_zip_code_prefix"].str.len().eq(5)).sum())

print(customers_clean["customer_zip_code_prefix"].head(10))

ZIP code length distribution:
customer_zip_code_prefix
5    99441
Name: count, dtype: int64
Unexpected ZIP code lengths: 0
0    14409
1    09790
2    01151
3    08775
4    13056
5    89254
6    04534
7    35182
8    81560
9    30575
Name: customer_zip_code_prefix, dtype: object


In [42]:
print("Original rows:", len(customers))
print("Cleaned rows:", len(customers_clean))

print("Missing values:", customers_clean.isnull().sum().sum())

print("Duplicate customer IDs:",
      customers_clean["customer_id"].duplicated().sum())

print("Invalid ZIP code lengths:",
      (~customers_clean["customer_zip_code_prefix"].str.len().eq(5)).sum())

Original rows: 99441
Cleaned rows: 99441
Missing values: 0
Duplicate customer IDs: 0
Invalid ZIP code lengths: 0


In [43]:
customers_clean.to_csv(
    os.path.join(PROCESSED_PATH, "customers_clean.csv"),
    index=False
)

In [4]:
products_clean = products.copy()

In [5]:
products_clean.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [6]:
print("Exact duplicate rows:",
      products_clean.duplicated().sum())

print("Duplicate product IDs:",
      products_clean.duplicated(
          subset=["product_id"]
      ).sum())

Exact duplicate rows: 0
Duplicate product IDs: 0


In [7]:
# Products missing catalog information
missing_catalog = products_clean[
    products_clean[
        ["product_category_name",
         "product_name_lenght",
         "product_description_lenght",
         "product_photos_qty"]
    ].isnull().any(axis=1)
]

print("Products missing catalog information:", len(missing_catalog))

# Products missing physical measurements
missing_dimensions = products_clean[
    products_clean[
        ["product_weight_g",
         "product_length_cm",
         "product_height_cm",
         "product_width_cm"]
    ].isnull().any(axis=1)
]

print("Products missing physical measurements:", len(missing_dimensions))

Products missing catalog information: 610
Products missing physical measurements: 2


In [8]:
catalog_columns = [
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

dimension_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

print("Products missing all 4 catalog fields:",
      products_clean[catalog_columns].isnull().all(axis=1).sum())

print("Products missing all 4 dimension fields:",
      products_clean[dimension_columns].isnull().all(axis=1).sum())

Products missing all 4 catalog fields: 610
Products missing all 4 dimension fields: 2


In [9]:
missing_dimensions

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
missing_dimensions[
    ["product_id", "product_category_name",
     "product_weight_g", "product_length_cm",
     "product_height_cm", "product_width_cm"]
]

,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN


In [12]:
print(
    products_clean.loc[
        products_clean["product_weight_g"].isnull(),
        [
            "product_id",
            "product_category_name",
            "product_name_lenght",
            "product_description_lenght",
            "product_photos_qty",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ].to_string(index=False)
)

                      product_id product_category_name  product_name_lenght  product_description_lenght  product_photos_qty  product_weight_g  product_length_cm  product_height_cm  product_width_cm
09ff539a621711667c43eba6a3bd8466                 bebes                 60.0                       865.0                 3.0               NaN                NaN                NaN               NaN
5eb564652db742ff8f28759cd8d2652a                   NaN                  NaN                         NaN                 NaN               NaN                NaN                NaN               NaN


In [13]:
products_clean.dtypes

product_id                     object
product_category_name          object
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object

In [14]:
numeric_columns = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for column in numeric_columns:
    print(
        column,
        "| Zero:", (products_clean[column] == 0).sum(),
        "| Negative:", (products_clean[column] < 0).sum()
    )

product_name_lenght | Zero: 0 | Negative: 0
product_description_lenght | Zero: 0 | Negative: 0
product_photos_qty | Zero: 0 | Negative: 0
product_weight_g | Zero: 4 | Negative: 0
product_length_cm | Zero: 0 | Negative: 0
product_height_cm | Zero: 0 | Negative: 0
product_width_cm | Zero: 0 | Negative: 0


In [15]:
zero_weight_products = products_clean[
    products_clean["product_weight_g"] == 0
]

print("Zero-weight products:", len(zero_weight_products))

print(
    zero_weight_products[
        [
            "product_id",
            "product_category_name",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ].to_string(index=False)
)

Zero-weight products: 4
                      product_id product_category_name  product_weight_g  product_length_cm  product_height_cm  product_width_cm
81781c0fed9fe1ad6e8c81fca1e1cb08       cama_mesa_banho               0.0               30.0               25.0              30.0
8038040ee2a71048d4bdbbdc985b69ab       cama_mesa_banho               0.0               30.0               25.0              30.0
36ba42dd187055e1fbe943b2d11430ca       cama_mesa_banho               0.0               30.0               25.0              30.0
e673e90efa65a5409ff4196c038bb5af       cama_mesa_banho               0.0               30.0               25.0              30.0


In [16]:
print("Original rows:", len(products))
print("Cleaned rows:", len(products_clean))

print("Missing values:", products_clean.isnull().sum().sum())

print("Duplicate product IDs:",
      products_clean["product_id"].duplicated().sum())

print("Zero-weight products:",
      (products_clean["product_weight_g"] == 0).sum())

print("Negative numeric values:",
      (products_clean[numeric_columns] < 0).sum().sum())

Original rows: 32951
Cleaned rows: 32951
Missing values: 2448
Duplicate product IDs: 0
Zero-weight products: 4
Negative numeric values: 0


In [17]:
products_clean.to_csv(
    os.path.join(PROCESSED_PATH, "products_clean.csv"),
    index=False
)

In [18]:
sellers_clean = sellers.copy()

print(sellers_clean.isnull().sum())

print("Exact duplicates:", sellers_clean.duplicated().sum())

print("Duplicate seller IDs:",
      sellers_clean["seller_id"].duplicated().sum())

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64
Exact duplicates: 0
Duplicate seller IDs: 0


In [19]:
sellers_clean.dtypes

seller_id                 object
seller_zip_code_prefix     int64
seller_city               object
seller_state              object
dtype: object

In [20]:
sellers_clean["seller_zip_code_prefix"] = (
    sellers_clean["seller_zip_code_prefix"].astype(str)
)

In [21]:
sellers_clean.dtypes

seller_id                 object
seller_zip_code_prefix    object
seller_city               object
seller_state              object
dtype: object

In [24]:
print("Missing ZIP codes:",
      sellers_clean["seller_zip_code_prefix"].isnull().sum())

print("ZIP codes with unexpected length:",
      (~sellers_clean["seller_zip_code_prefix"].str.len().eq(5)).sum())

print("Unique ZIP codes:",
      sellers_clean["seller_zip_code_prefix"].nunique())

print("Sample ZIP codes:")
print(sellers_clean["seller_zip_code_prefix"].head(10))

Missing ZIP codes: 0
ZIP codes with unexpected length: 0
Unique ZIP codes: 2246
Sample ZIP codes:
0    13023
1    13844
2    20031
3    04195
4    12914
5    20920
6    55325
7    16304
8    01529
9    80310
Name: seller_zip_code_prefix, dtype: object


In [25]:
sellers_clean["seller_zip_code_prefix"].str.len().value_counts().sort_index()

seller_zip_code_prefix
5    3095
Name: count, dtype: int64

In [26]:
sellers_clean["seller_zip_code_prefix"] = (
    sellers_clean["seller_zip_code_prefix"].str.zfill(5)
)

In [27]:
print("ZIP code length distribution:")
print(sellers_clean["seller_zip_code_prefix"].str.len().value_counts())

print("Unexpected ZIP code lengths:",
      (~sellers_clean["seller_zip_code_prefix"].str.len().eq(5)).sum())

print(sellers_clean["seller_zip_code_prefix"].head(10))

ZIP code length distribution:
seller_zip_code_prefix
5    3095
Name: count, dtype: int64
Unexpected ZIP code lengths: 0
0    13023
1    13844
2    20031
3    04195
4    12914
5    20920
6    55325
7    16304
8    01529
9    80310
Name: seller_zip_code_prefix, dtype: object


In [28]:
print("Original rows:", len(sellers))
print("Cleaned rows:", len(sellers_clean))

print("Missing values:", sellers_clean.isnull().sum().sum())

print("Duplicate seller IDs:",
      sellers_clean["seller_id"].duplicated().sum())

print("Invalid ZIP code lengths:",
      (~sellers_clean["seller_zip_code_prefix"].str.len().eq(5)).sum())

Original rows: 3095
Cleaned rows: 3095
Missing values: 0
Duplicate seller IDs: 0
Invalid ZIP code lengths: 0


In [29]:
sellers_clean.to_csv(
    os.path.join(PROCESSED_PATH, "sellers_clean.csv"),
    index=False
)

In [30]:
payments_clean = payments.copy()

In [31]:
payments_clean.isnull().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [32]:
print("Exact duplicate rows:",
      payments_clean.duplicated().sum())

print("Payment order ID missing:",
      payments_clean["order_id"].isnull().sum())

Exact duplicate rows: 0
Payment order ID missing: 0


In [33]:
payments_clean.dtypes

order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object

In [34]:
print("Zero payment values:",
      (payments_clean["payment_value"] == 0).sum())

print("Negative payment values:",
      (payments_clean["payment_value"] < 0).sum())

print("Zero installments:",
      (payments_clean["payment_installments"] == 0).sum())

print("Negative installments:",
      (payments_clean["payment_installments"] < 0).sum())

Zero payment values: 9
Negative payment values: 0
Zero installments: 2
Negative installments: 0


In [35]:
zero_payment_records = payments_clean[
    payments_clean["payment_value"] == 0
]

print("Zero payment records:", len(zero_payment_records))

print(
    zero_payment_records[
        ["order_id", "payment_sequential", "payment_type",
         "payment_installments", "payment_value"]
    ].to_string(index=False)
)

Zero payment records: 9
                        order_id  payment_sequential payment_type  payment_installments  payment_value
8bcbe01d44d147f901cd3192671144db                   4      voucher                     1            0.0
fa65dad1b0e818e3ccc5cb0e39231352                  14      voucher                     1            0.0
6ccb433e00daae1283ccc956189c82ae                   4      voucher                     1            0.0
4637ca194b6387e2d538dc89b124b0ee                   1  not_defined                     1            0.0
00b1cb0320190ca0daa2c88b35206009                   1  not_defined                     1            0.0
45ed6e85398a87c253db47c2d9f48216                   3      voucher                     1            0.0
fa65dad1b0e818e3ccc5cb0e39231352                  13      voucher                     1            0.0
c8c528189310eaa44a745b8d9d26908b                   1  not_defined                     1            0.0
b23878b3e8eb4d25a158f57d96331b18                 

In [36]:
zero_installment_records = payments_clean[
    payments_clean["payment_installments"] == 0
]

print(
    zero_installment_records[
        ["order_id", "payment_sequential", "payment_type",
         "payment_installments", "payment_value"]
    ].to_string(index=False)
)

                        order_id  payment_sequential payment_type  payment_installments  payment_value
744bade1fcf9ff3f31d860ace076d422                   2  credit_card                     0          58.69
1a57108394169c0b47d8f876acc9ba2d                   2  credit_card                     0         129.94


In [37]:
print(payments_clean["payment_type"].value_counts())

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


In [38]:
print(payments_clean["payment_installments"].value_counts().sort_index())

payment_installments
0         2
1     52546
2     12413
3     10461
4      7098
5      5239
6      3920
7      1626
8      4268
9       644
10     5328
11       23
12      133
13       16
14       15
15       74
16        5
17        8
18       27
20       17
21        3
22        1
23        1
24       18
Name: count, dtype: int64


In [40]:
unmatched_orders = payments_clean[
    ~payments_clean["order_id"].isin(orders["order_id"])
]

print("Payments with unmatched order IDs:", len(unmatched_orders))

Payments with unmatched order IDs: 0


In [41]:
print("Original rows:", len(payments))
print("Cleaned rows:", len(payments_clean))

print("Missing values:", payments_clean.isnull().sum().sum())

print("Exact duplicates:", payments_clean.duplicated().sum())

print("Unmatched order IDs:",
      (~payments_clean["order_id"].isin(orders["order_id"])).sum())

Original rows: 103886
Cleaned rows: 103886
Missing values: 0
Exact duplicates: 0
Unmatched order IDs: 0


In [42]:
payments_clean.to_csv(
    os.path.join(PROCESSED_PATH, "payments_clean.csv"),
    index=False
)

In [3]:
reviews_clean = reviews.copy()

In [4]:
reviews_clean.isnull().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [5]:
print("Exact duplicate rows:",
      reviews_clean.duplicated().sum())

print("Missing order IDs:",
      reviews_clean["order_id"].isnull().sum())

Exact duplicate rows: 0
Missing order IDs: 0


In [6]:
reviews_clean["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [8]:
print("Invalid review scores:",
      (~reviews_clean["review_score"].between(1, 5)).sum())

Invalid review scores: 0


In [9]:
reviews_clean.dtypes

review_id                  object
order_id                   object
review_score                int64
review_comment_title       object
review_comment_message     object
review_creation_date       object
review_answer_timestamp    object
dtype: object

In [10]:
date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

for column in date_columns:
    reviews_clean[column] = pd.to_datetime(
        reviews_clean[column],
        errors="coerce"
    )

In [11]:
print(reviews_clean[date_columns].dtypes)

print("\nMissing dates after conversion:")
print(reviews_clean[date_columns].isnull().sum())

review_creation_date       datetime64[ns]
review_answer_timestamp    datetime64[ns]
dtype: object

Missing dates after conversion:
review_creation_date       0
review_answer_timestamp    0
dtype: int64


In [12]:
invalid_review_dates = reviews_clean[
    reviews_clean["review_answer_timestamp"] <
    reviews_clean["review_creation_date"]
]

print("Invalid review date records:", len(invalid_review_dates))

Invalid review date records: 0


In [13]:
unmatched_reviews = reviews_clean[
    ~reviews_clean["order_id"].isin(orders["order_id"])
]

print("Reviews with unmatched order IDs:", len(unmatched_reviews))

Reviews with unmatched order IDs: 0


In [14]:
reviews_clean.to_csv(
    os.path.join(PROCESSED_PATH, "reviews_clean.csv"),
    index=False
)

In [15]:
geolocation_clean = geolocation.copy()

In [16]:
geolocation_clean.isnull().sum()

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

In [17]:
print("Exact duplicate rows:",
      geolocation_clean.duplicated().sum())

Exact duplicate rows: 261831


In [18]:
geolocation_clean.dtypes

geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

In [19]:
geolocation_clean["geolocation_zip_code_prefix"].astype(str).str.len().value_counts().sort_index()

geolocation_zip_code_prefix
4    245733
5    754430
Name: count, dtype: int64

In [20]:
print(
    "Unique ZIP prefixes:",
    geolocation_clean["geolocation_zip_code_prefix"].nunique()
)

Unique ZIP prefixes: 19015


In [21]:
print(
    "Unique latitude-longitude combinations:",
    geolocation_clean[
        ["geolocation_zip_code_prefix",
         "geolocation_lat",
         "geolocation_lng"]
    ].drop_duplicates().shape[0]
)

Unique latitude-longitude combinations: 720154


In [22]:
geolocation_clean["geolocation_zip_code_prefix"] = (
    geolocation_clean["geolocation_zip_code_prefix"].astype(str)
)

In [23]:
geolocation_clean["geolocation_zip_code_prefix"] = (
    geolocation_clean["geolocation_zip_code_prefix"].str.zfill(5)
)

In [24]:
geolocation_clean["geolocation_zip_code_prefix"].str.len().value_counts()

geolocation_zip_code_prefix
5    1000163
Name: count, dtype: int64

In [25]:
geolocation_clean = geolocation_clean.drop_duplicates().copy()

In [26]:
print("Rows after removing duplicates:", len(geolocation_clean))
print("Remaining exact duplicates:", geolocation_clean.duplicated().sum())

Rows after removing duplicates: 738332
Remaining exact duplicates: 0


In [27]:
invalid_coordinates = geolocation_clean[
    (geolocation_clean["geolocation_lat"] < -90) |
    (geolocation_clean["geolocation_lat"] > 90) |
    (geolocation_clean["geolocation_lng"] < -180) |
    (geolocation_clean["geolocation_lng"] > 180)
]

print("Invalid coordinates:", len(invalid_coordinates))

Invalid coordinates: 0


In [28]:
suspicious_coordinates = geolocation_clean[
    (geolocation_clean["geolocation_lat"] < -35) |
    (geolocation_clean["geolocation_lat"] > 6) |
    (geolocation_clean["geolocation_lng"] < -75) |
    (geolocation_clean["geolocation_lng"] > -30)
]

print("Suspicious geographical records:", len(suspicious_coordinates))

Suspicious geographical records: 25


In [29]:
suspicious_coordinates[
    [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state"
    ]
].head(30)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.008978,-15.536867,bom retiro da esperanca,SP
513631,28165,41.614052,-8.411675,vila nova de campos,RJ
513754,28155,42.439286,13.820214,santa maria,RJ
514429,28333,38.381672,-6.328200,raposo,RJ
516682,28595,43.684961,-7.411080,portela,RJ
538512,29654,29.409252,-98.484121,santo antônio do canaã,ES
538557,29654,21.657547,-101.466766,santo antonio do canaa,ES
585242,35179,25.995203,-98.078544,santana do paraíso,MG
585260,35179,25.995245,-98.078533,santana do paraiso,MG
695377,45936,38.323939,-6.775035,itabatan,BA


In [30]:
state_counts = (
    geolocation_clean
    .groupby("geolocation_zip_code_prefix")["geolocation_state"]
    .nunique()
)

multiple_states = state_counts[state_counts > 1]

print("ZIP prefixes associated with multiple states:",
      len(multiple_states))

ZIP prefixes associated with multiple states: 8


In [31]:
multiple_state_prefixes = multiple_states.index

multiple_state_records = geolocation_clean[
    geolocation_clean["geolocation_zip_code_prefix"].isin(
        multiple_state_prefixes
    )
].sort_values(
    ["geolocation_zip_code_prefix", "geolocation_state"]
)

print("Records for ZIP prefixes with multiple states:",
      len(multiple_state_records))

display(multiple_state_records)

Records for ZIP prefixes with multiple states: 572


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
22261,02116,-23.515978,-46.582170,são paulo,RN
21728,02116,-23.522700,-46.587546,sao paulo,SP
21759,02116,-23.518459,-46.584128,sao paulo,SP
22060,02116,-23.519065,-46.584610,sao paulo,SP
22193,02116,-23.519739,-46.585151,são paulo,SP
...,...,...,...,...,...
847857,80630,-25.471820,-49.281597,curitiba,PR
847872,80630,-25.469776,-49.273230,curitiba,PR
847875,80630,-25.466519,-49.268213,curitiba,PR
847897,80630,-25.473179,-49.282526,curitiba,PR


In [32]:
state_frequency = (
    geolocation_clean
    .groupby(["geolocation_zip_code_prefix", "geolocation_state"])
    .size()
    .reset_index(name="record_count")
)

dominant_state = (
    state_frequency
    .sort_values("record_count", ascending=False)
    .drop_duplicates(
        subset=["geolocation_zip_code_prefix"]
    )
)

print(dominant_state)

      geolocation_zip_code_prefix geolocation_state  record_count
9540                        38400                MG           779
8732                        35500                MG           751
4503                        11680                SP           727
4530                        11740                SP           678
9019                        36400                MG           627
...                           ...               ...           ...
9925                        39999                MG             1
9919                        39955                MG             1
17939                       90619                RS             1
4473                        11620                SP             1
9891                        39866                MG             1

[19015 rows x 3 columns]


In [33]:
exceptional_records = multiple_state_records[
    ~multiple_state_records["geolocation_state"].isin(
        dominant_state.set_index(
            "geolocation_zip_code_prefix"
        )["geolocation_state"]
    )
]

print("Exceptional records:", len(exceptional_records))

Exceptional records: 0


In [34]:
exceptional_records = multiple_state_records.merge(
    dominant_state[
        ["geolocation_zip_code_prefix", "geolocation_state"]
    ],
    on="geolocation_zip_code_prefix",
    how="left",
    suffixes=("", "_dominant")
)

exceptional_records = exceptional_records[
    exceptional_records["geolocation_state"] !=
    exceptional_records["geolocation_state_dominant"]
]

print("Exceptional records:", len(exceptional_records))

Exceptional records: 8


In [35]:
display(exceptional_records)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,geolocation_state_dominant
0,02116,-23.515978,-46.582170,são paulo,RN,SP
11,04011,-23.578707,-46.645779,sao paulo,AC,SP
81,21550,-22.857861,-43.352613,rio de janeiro,AC,RJ
226,23056,-22.919164,-43.611097,rio de janeiro,AC,RJ
257,72915,-12.408760,-46.428287,taguatinga,DF,GO
342,78557,-11.875142,-55.503020,nova brasilandia d'oeste,RO,MT
494,79750,-22.242062,-53.343159,nova andradina,RS,MS
571,80630,-26.757371,-48.675738,balneario de picarras,SC,PR


In [36]:
print("Original rows:", len(geolocation))
print("Cleaned rows:", len(geolocation_clean))

print("Missing values:",
      geolocation_clean.isnull().sum().sum())

print("Exact duplicates:",
      geolocation_clean.duplicated().sum())

print("Unique ZIP prefixes:",
      geolocation_clean["geolocation_zip_code_prefix"].nunique())

print("Invalid ZIP lengths:",
      (geolocation_clean["geolocation_zip_code_prefix"].str.len() != 5).sum())

print("Suspicious geographical records:",
      len(suspicious_coordinates))

print("Exceptional state records:",
      len(exceptional_records))

Original rows: 1000163
Cleaned rows: 738332
Missing values: 0
Exact duplicates: 0
Unique ZIP prefixes: 19015
Invalid ZIP lengths: 0
Suspicious geographical records: 25
Exceptional state records: 8


In [37]:
geolocation_clean.to_csv(
    os.path.join(PROCESSED_PATH, "geolocation_clean.csv"),
    index=False
)

In [38]:
category_translation_clean = category_translation.copy()

In [39]:
print(category_translation_clean.isnull().sum())

print("Exact duplicates:",
      category_translation_clean.duplicated().sum())

print("Duplicate Portuguese categories:",
      category_translation_clean[
          "product_category_name"
      ].duplicated().sum())

print("Duplicate English categories:",
      category_translation_clean[
          "product_category_name_english"
      ].duplicated().sum())

display(category_translation_clean.head())

product_category_name            0
product_category_name_english    0
dtype: int64
Exact duplicates: 0
Duplicate Portuguese categories: 0
Duplicate English categories: 0


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [40]:
print("Original rows:", len(category_translation))
print("Cleaned rows:", len(category_translation_clean))

print("Missing values:",
      category_translation_clean.isnull().sum().sum())

print("Exact duplicates:",
      category_translation_clean.duplicated().sum())

Original rows: 71
Cleaned rows: 71
Missing values: 0
Exact duplicates: 0


In [41]:
category_translation_clean.to_csv(
    os.path.join(PROCESSED_PATH, "category_translation_clean.csv"),
    index=False
)

In [42]:
print("Files in Processed folder:\n")

for file in os.listdir(PROCESSED_PATH):
    if file.endswith(".csv"):
        print(file)

Files in Processed folder:

category_translation_clean.csv
customers_clean.csv
geolocation_clean.csv
order_items_clean.csv
payments_clean.csv
products_clean.csv
reviews_clean.csv
sellers_clean.csv


In [43]:
orders_clean.to_csv(
    os.path.join(PROCESSED_PATH, "orders_clean.csv"),
    index=False
)

NameError: name 'orders_clean' is not defined

In [44]:
orders_clean = orders.copy()

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders_clean[column] = pd.to_datetime(
        orders_clean[column],
        errors="coerce"
    )

In [45]:
orders_clean.to_csv(
    os.path.join(PROCESSED_PATH, "orders_clean.csv"),
    index=False
)

In [46]:
csv_files = [
    file for file in os.listdir(PROCESSED_PATH)
    if file.endswith(".csv")
]

print("Total cleaned CSV files:", len(csv_files))

for file in sorted(csv_files):
    print(file)

Total cleaned CSV files: 9
category_translation_clean.csv
customers_clean.csv
geolocation_clean.csv
order_items_clean.csv
orders_clean.csv
payments_clean.csv
products_clean.csv
reviews_clean.csv
sellers_clean.csv


In [48]:
orders_saved = pd.read_csv(
    os.path.join(PROCESSED_PATH, "orders_clean.csv")
)

print(orders_saved.shape)
print(orders_saved.dtypes)

(99441, 8)
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object


In [49]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders_saved[column] = pd.to_datetime(
        orders_saved[column],
        errors="coerce"
    )

In [50]:
print("Original rows:", len(orders))
print("Saved rows:", len(orders_saved))

print("Original missing values:",
      orders.isnull().sum().sum())

print("Saved missing values:",
      orders_saved.isnull().sum().sum())

print("Duplicate order IDs:",
      orders_saved["order_id"].duplicated().sum())

print("Date columns:")
print(orders_saved[date_columns].dtypes)

Original rows: 99441
Saved rows: 99441
Original missing values: 4908
Saved missing values: 4908
Duplicate order IDs: 0
Date columns:
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [51]:
order_items_saved = pd.read_csv(os.path.join(PROCESSED_PATH, "order_items_clean.csv"))
customers_saved = pd.read_csv(os.path.join(PROCESSED_PATH, "customers_clean.csv"))
products_saved = pd.read_csv(os.path.join(PROCESSED_PATH, "products_clean.csv"))
sellers_saved = pd.read_csv(os.path.join(PROCESSED_PATH, "sellers_clean.csv"))
payments_saved = pd.read_csv(os.path.join(PROCESSED_PATH, "payments_clean.csv"))
reviews_saved = pd.read_csv(os.path.join(PROCESSED_PATH, "reviews_clean.csv"))
geolocation_saved = pd.read_csv(os.path.join(PROCESSED_PATH, "geolocation_clean.csv"))
category_translation_saved = pd.read_csv(os.path.join(PROCESSED_PATH, "category_translation_clean.csv"))

In [52]:
datasets = {
    "Order Items": (order_items, order_items_saved),
    "Customers": (customers, customers_saved),
    "Products": (products, products_saved),
    "Sellers": (sellers, sellers_saved),
    "Payments": (payments, payments_saved),
    "Reviews": (reviews, reviews_saved),
    "Geolocation": (geolocation, geolocation_saved),
    "Category Translation": (category_translation, category_translation_saved)
}

for name, (original, saved) in datasets.items():
    print("\n", name)
    print("Original rows:", len(original))
    print("Saved rows:", len(saved))
    print("Original missing values:", original.isnull().sum().sum())
    print("Saved missing values:", saved.isnull().sum().sum())


 Order Items
Original rows: 112650
Saved rows: 112650
Original missing values: 0
Saved missing values: 0

 Customers
Original rows: 99441
Saved rows: 99441
Original missing values: 0
Saved missing values: 0

 Products
Original rows: 32951
Saved rows: 32951
Original missing values: 2448
Saved missing values: 2448

 Sellers
Original rows: 3095
Saved rows: 3095
Original missing values: 0
Saved missing values: 0

 Payments
Original rows: 103886
Saved rows: 103886
Original missing values: 0
Saved missing values: 0

 Reviews
Original rows: 99224
Saved rows: 99224
Original missing values: 145903
Saved missing values: 145903

 Geolocation
Original rows: 1000163
Saved rows: 738332
Original missing values: 0
Saved missing values: 0

 Category Translation
Original rows: 71
Saved rows: 71
Original missing values: 0
Saved missing values: 0
